# 03 Logistic Regression

We train models on the training students and compare them on the validation students.
The test set is not used here. It is used only once, at the very end.

In [ ]:
import sys
sys.path.append("..")

from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss

from src.data import load_data, split_students
from src.features import build_features

# 1. Load the data, split the students, add the clues
df = build_features(split_students(load_data()))

# 2. Gender and free school meals as 0/1 columns (so feature selection can try them)
df["female"] = (df["Gender"] == 1).astype(int)
df["male"] = (df["Gender"] == 2).astype(int)
df["free_school_meals"] = (df["PremiumPupil"] == 1).astype(int)
df["fsm_known"] = df["PremiumPupil"].notna().astype(int)

# 3. Training and validation students (test is only used in notebook 05)
train = df[df["split"] == "train"]
val = df[df["split"] == "val"]

print("Training answers:  ", f"{len(train):,}")
print("Validation answers:", f"{len(val):,}")

Training answers:   977,349
Validation answers: 200,546


### Question difficulty
How hard is each question? We count the share of training students who got it right.

In [2]:
subjects = pd.read_csv("../data/metadata/subject_metadata.csv")
name_of = dict(zip(subjects["SubjectId"], subjects["Name"]))

questions = (
    train.groupby("QuestionId")
         .agg(students=("IsCorrect", "size"), pct_correct=("IsCorrect", "mean"),
              topic=("skill_L3", "first"))
         .assign(pct_correct=lambda t: (t["pct_correct"] * 100).round(1),
                 topic=lambda t: t["topic"].map(name_of))
         .sort_values("pct_correct")
)
questions = questions[questions["students"] >= 50]   # ignore questions answered by very few students

print("Hardest 5 questions:")
print(questions.head(5).to_string())
print("\nEasiest 5 questions:")
print(questions.tail(5).to_string())

Hardest 5 questions:
            students  pct_correct                                  topic
QuestionId                                                              
931             1223         15.6           Properties of Quadrilaterals
155             1183         16.0                    Angles in Triangles
944              135         17.0        Prime Numbers and Prime Factors
718              474         17.1                            Enlargement
784              317         17.7  Length, Area and Volume Scale Factors

Easiest 5 questions:
            students  pct_correct                               topic
QuestionId                                                           
312             1982         86.2  Mental Multiplication and Division
828             1999         87.1                 Squares, Cubes, etc
825             1707         89.8  Mental Multiplication and Division
422              106         91.5   Factors and Highest Common Factor
924             1664      

**Takeaway:** Questions range from very hard (16% of training students right, e.g. properties of quadrilaterals and angles in triangles) to very easy (93% right, mostly mental multiplication and division). We only show questions answered by at least 50 students, because a few questions have only 2–3 answers. This big spread is why question difficulty is a useful clue.

## 1. Baselines
Simple guesses that any real model must beat.
- **Majority:** always predict "correct".
- **Question difficulty:** predict each question's training correct rate.

In [3]:
results = []

def score(name, y_true, prob):
    """AUC and accuracy for one predictor on validation."""
    results.append({
        "model": name,
        "val AUC": round(roc_auc_score(y_true, prob), 4),
        "val accuracy": round(accuracy_score(y_true, prob >= 0.5), 4),
    })

score("Majority (always correct)", val["IsCorrect"], np.ones(len(val)))
score("Question difficulty", val["IsCorrect"], val["q_correct_rate"])
pd.DataFrame(results)

,model,val AUC,val accuracy
0,Majority (always correct),0.5000,0.5472
1,Question difficulty,0.6754,0.6302


**Takeaway:** If we always guess "correct", we get 54.7% right, but AUC is only 0.5 because every answer gets the same score. If we only use how hard the question is, we get 63.0% right and AUC 0.675. Our models must do better than this.

## 2. Forward selection
We let the data choose the features. We start with no features and, at each step, add the one that raises validation AUC the most. We stop when the best one adds less than 0.001.
We can choose from 13 features: the student's history (overall and on each topic level), question difficulty, gender and free school meals.

In [4]:
# ---------- All the features we can choose from ----------
all_features = [
    "acc_prev", "n_prev",              # student history (all questions)
    "acc_prev_L1", "n_prev_L1",        # history on the level-1 topic
    "acc_prev_L2", "n_prev_L2",        # history on the level-2 topic
    "acc_prev_L3", "n_prev_L3",        # history on the level-3 topic
    "q_correct_rate",                  # question difficulty
    "female", "male",                  # gender
    "free_school_meals", "fsm_known",  # free school meals
]

# ---------- A helper: train a model and return its validation AUC ----------
def val_auc(features):
    model = make_pipeline(StandardScaler(), LogisticRegression())
    model.fit(train[features], train["IsCorrect"])
    prob = model.predict_proba(val[features])[:, 1]
    return roc_auc_score(val["IsCorrect"], prob)

# ---------- Forward selection ----------
selected = []        # features we keep
best_auc = 0.5       # no features = a coin flip
tries = []           # EVERY feature we try, at every step

for step in range(1, len(all_features) + 1):

    best_feature = None
    best_new_auc = 0
    for feature in all_features:
        if feature in selected:
            continue
        auc = val_auc(selected + [feature])
        tries.append({"step": step, "feature": feature, "gain": auc - best_auc})   # remember every try
        if auc > best_new_auc:
            best_new_auc = auc
            best_feature = feature

    gain = best_new_auc - best_auc
    print(f"Step {step}: best = {best_feature}, AUC {best_new_auc:.4f}, gain {gain:+.4f} ->",
          "KEEP" if gain >= 0.001 else "STOP")

    if gain < 0.001:
        break
    selected.append(best_feature)
    best_auc = best_new_auc

# ---------- Table: how much each feature would add at each step ----------
gains = pd.DataFrame(tries).pivot(index="feature", columns="step", values="gain").round(4)
gains.columns = [f"step {s}" for s in gains.columns]
gains["kept"] = ["✓ step " + str(selected.index(f) + 1) if f in selected else "" for f in gains.index]
gains = gains.loc[selected + [f for f in all_features if f not in selected]]   # kept ones on top

print("\nFeatures we keep:", selected)
gains.astype(object).fillna("—")

Step 1: best = acc_prev, AUC 0.7011, gain +0.2011 -> KEEP


Step 2: best = q_correct_rate, AUC 0.7716, gain +0.0705 -> KEEP


Step 3: best = acc_prev_L2, AUC 0.7759, gain +0.0043 -> KEEP


Step 4: best = n_prev, AUC 0.7764, gain +0.0005 -> STOP

Features we keep: ['acc_prev', 'q_correct_rate', 'acc_prev_L2']


,step 1,step 2,step 3,step 4,kept
feature,,,,,
acc_prev,0.2011,—,—,—,✓ step 1
q_correct_rate,0.1754,0.0705,—,—,✓ step 2
acc_prev_L2,0.1833,0.0067,0.0043,—,✓ step 3
n_prev,0.0316,0.0006,0.0004,0.0005,
acc_prev_L1,0.2001,0.0046,0.0027,0.0005,
n_prev_L1,0.0202,0.0004,0.0,-0.0,
n_prev_L2,0.0073,0.0003,0.0,0.0,
acc_prev_L3,0.1649,0.0067,0.0036,0.0004,
n_prev_L3,0.0029,0.0002,0.0,0.0,


**Takeaway:** Forward selection keeps three features: the student's accuracy so far (+0.20 AUC), question difficulty (+0.07) and accuracy on the level-2 topic (+0.004). After that, the best new feature (`n_prev`) adds only +0.0005, so we stop. Many features look useful alone but stop helping once the model knows the student's accuracy. For example, gender and free school meals add up to +0.02 alone, but 0 after the first step.

### How much does each feature matter?
**AUC alone:** how good the feature is by itself (0.5 = useless).
**Effect in final model:** for a kept feature, how much AUC drops if we remove it; for the others, how much AUC would rise if we added it.

In [5]:
final_auc = val_auc(selected)

rows = []
for feature in all_features:
    alone = val_auc([feature])
    if feature in selected:
        effect = final_auc - val_auc([f for f in selected if f != feature])   # take it out
    else:
        effect = val_auc(selected + [feature]) - final_auc                  # put it in
    rows.append({"feature": feature,
                 "AUC alone": round(alone, 4),
                 "effect in final model": round(effect, 4),
                 "selected": feature in selected})

pd.DataFrame(rows).sort_values("AUC alone", ascending=False)

,feature,AUC alone,effect in final model,selected
0,acc_prev,0.7011,0.0196,True
2,acc_prev_L1,0.7001,0.0005,False
4,acc_prev_L2,0.6833,0.0043,True
8,q_correct_rate,0.6754,0.0681,True
6,acc_prev_L3,0.6649,0.0004,False
1,n_prev,0.5316,0.0005,False
11,free_school_meals,0.5217,0.0000,False
3,n_prev_L1,0.5202,-0.0000,False
10,male,0.5189,-0.0000,False
12,fsm_known,0.5188,0.0000,False


**Takeaway:** On its own, the student's accuracy so far is the best feature (AUC 0.70). In the final model, question difficulty matters most: without it, AUC drops by 0.068. Overall accuracy and level-2 topic accuracy share the "how good is this student" job, so each one costs less when removed. The answer counts, gender and free school meals are close to useless alone (AUC about 0.5) and add nothing to the final model.

## 3. Best subset selection
Forward selection is fast but greedy: once it picks a feature, it never changes its mind. **Best subset selection** tries every combination of up to 3 features (377 models) and finds the best one of each size. We use it to check forward selection's choice.

In [21]:
MAX_SIZE = 3

subset_results = []
for size in range(1, MAX_SIZE + 1):                 
    for combo in combinations(all_features, size):
        features = list(combo)
        subset_results.append({"size": size,
                               "features": ", ".join(features),
                               "val AUC": val_auc(features)})
subset_results = pd.DataFrame(subset_results)

print("Combinations tried:", len(subset_results))
best_per_size = (subset_results.sort_values("val AUC", ascending=False)
                               .groupby("size").head(1)
                               .sort_values("size"))
best_per_size["val AUC"] = best_per_size["val AUC"].round(4)
print("Forward selection picked:", ", ".join(selected), f"(val AUC {best_auc:.4f})")

# ---------- Use the best combination of the largest size ----------
best_row = best_per_size.iloc[-1]                  
selected = best_row["features"].split(", ")
best_auc = best_row["val AUC"]
print("Best subset picked:      ", ", ".join(selected), f"(val AUC {best_auc:.4f})")

best_per_size

Combinations tried: 377
Forward selection picked: acc_prev, q_correct_rate, acc_prev_L2 (val AUC 0.7759)
Best subset picked:       acc_prev, acc_prev_L2, q_correct_rate (val AUC 0.7759)


,size,features,val AUC
0,1,acc_prev,0.7011
20,2,"acc_prev, q_correct_rate",0.7716
124,3,"acc_prev, acc_prev_L2, q_correct_rate",0.7759


**Takeaway:** We tried all 377 combinations of up to 3 features. The best combination of each size is exactly what forward selection picked: accuracy so far (1 feature), plus question difficulty (2 features), plus level-2 topic accuracy (3 features, AUC 0.7759). So forward selection found the best features here, even though it does not try every combination.

## 4. Tuning C (regularization)

`C` controls how strongly the model is kept simple: a small C keeps the weights small (less overfitting), a large C lets them grow freely.
We try C from 0.0001 to 100 with the features from forward selection and keep the smallest C with the best validation log loss. We use log loss because our project depends on good probabilities, not just ranking.

In [ ]:
final_features = selected
rows = []
for C in [0.0001, 0.001, 0.01, 0.1, 1, 10, 100]:
    model = make_pipeline(StandardScaler(), LogisticRegression(C=C))
    model.fit(train[final_features], train["IsCorrect"])
    prob = model.predict_proba(val[final_features])[:, 1]
    rows.append({"C": C,
                 "val AUC": roc_auc_score(val["IsCorrect"], prob),
                 "val log loss": log_loss(val["IsCorrect"], prob)})

tuning = pd.DataFrame(rows)

best_loss = tuning["val log loss"].min()
best_C = tuning[tuning["val log loss"] <= best_loss + 0.00001]["C"].min()

print("Chosen C:", best_C)
tuning.round(5)

Chosen C: 0.01


,C,val AUC,val log loss
0,0.0001,0.77591,0.56794
1,0.0010,0.77592,0.56748
2,0.0100,0.77592,0.56746
3,0.1000,0.77592,0.56746
4,1.0000,0.77592,0.56746
5,10.0000,0.77592,0.56746
6,100.0000,0.77592,0.56746


**Takeaway:** The result is almost the same for every C (AUC 0.7759). We have about 977,000 training answers and only 3 features, so the model cannot overfit, and C barely matters. Only the smallest C (0.0001) is a little worse. C = 0.01 is the smallest value with the best log loss (0.5675), so we use it.

## 5. Final logistic regression
The selected features and C, trained on the training students only.

In [31]:
# ---------- 1. Train the final model ----------
final_model = make_pipeline(StandardScaler(), LogisticRegression(C=best_C))
final_model.fit(train[final_features], train["IsCorrect"])
val_prob = final_model.predict_proba(val[final_features])[:, 1]

# ---------- 2. Table: scores (the two simple guesses + our model) ----------
y = val["IsCorrect"]
always_correct = np.ones(len(val))
difficulty = val["q_correct_rate"]

scores = pd.DataFrame({
    "model":    ["Majority", "Question difficulty", "Logistic regression"],
    "val AUC":  [roc_auc_score(y, always_correct),
                 roc_auc_score(y, difficulty),
                 roc_auc_score(y, val_prob)],
    "val accuracy": [accuracy_score(y, always_correct >= 0.5),
                     accuracy_score(y, difficulty >= 0.5),
                     accuracy_score(y, val_prob >= 0.5)],
}).round(4)
print("Scores on validation students:")
display(scores)

# ---------- 3. Table: what the model learned ----------
weights = pd.DataFrame({
    "feature": final_features,
    "weight":  final_model[-1].coef_[0].round(3),
}).sort_values("weight", ascending=False)
print("What the model learned:")
weights

Scores on validation students:


,model,val AUC,val accuracy
0,Majority,0.5000,0.5472
1,Question difficulty,0.6754,0.6302
2,Logistic regression,0.7759,0.7085


What the model learned:


,feature,weight
2,q_correct_rate,0.724
0,acc_prev,0.646
1,acc_prev_L2,0.279


**Takeaway:** The final model gets AUC 0.776 and 70.9% right on validation, much better than always guessing "correct" (54.7%) or using only question difficulty (63.0%). The most important feature is question difficulty (weight 0.724), then the student's accuracy so far (0.646) and their accuracy on the level-2 topic (0.279). All weights are positive: an easier question and a stronger student both mean "more likely correct".

## 6. Save predictions
We save predictions for validation and test so the evaluation notebook can compare all models without retraining.
We do **not** score the test set here. It is scored once, in the evaluation notebook.

In [33]:
to_predict = df[df["split"].isin(["val", "test"])]

preds = pd.DataFrame({
    "AnswerId": to_predict["AnswerId"],
    "split": to_predict["split"],
    "p_correct": final_model.predict_proba(to_predict[final_features])[:, 1],
})
preds.to_csv("../results/preds_logreg.csv", index=False)
print(preds.shape)
preds.head()

(405378, 3)


,AnswerId,split,p_correct
333,1160716,val,0.736821
334,1114855,val,0.781718
335,376576,val,0.909658
336,968084,val,0.775402
337,185605,val,0.892014


**Takeaway:** We saved 405,378 predictions (200,546 validation + 204,832 test) to `results/preds_logreg.csv`. The neural network saves the same format, so notebook 05 can compare them answer by answer.